In [ ]:
import os
# os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
import random

In [ ]:
def set_seed(seed: int):
    random.seed(seed) # Python
    np.random.seed(seed) # NumPy
    torch.manual_seed(seed) # PyTorch (CPU)
    torch.cuda.manual_seed(seed) # PyTorch (GPU)
    torch.cuda.manual_seed_all(seed) # PyTorch (GPU)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
def load_karate():
    G = nx.karate_club_graph()
    N = G.number_of_nodes()
    y = np.array([0 if G.nodes[i]["club"] == "Mr.␣Hi" else 1 for i in range(N)],
    dtype=np.int64)
    A = nx.to_numpy_array(G, dtype=np.float32) # Adjacency matrix
    X = np.eye(N, dtype=np.float32) # One-hot node features
    np.fill_diagonal(A, 1.0)
    # Symmetric normalization: D^{-1/2} A D^{-1/2}
    deg = np.sum(A, axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(deg + 1e-8))
    A_norm = (D_inv_sqrt @ A @ D_inv_sqrt)
    return torch.from_numpy(X), torch.from_numpy(A_norm), torch.from_numpy(y)

In [ ]:
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.ReLU = nn.ReLU()
    def forward(self, X, A_norm):
        # X : (N, D)
        # A_norm : (N, N)
        # X - H :
        # W :
        return self.ReLU(A_norm @ self.linear(X))

class GCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.l1 = GCNLayer(in_dim, hidden_dim)
        self.l2 = nn.Linear(hidden_dim, out_dim)
        #레이어에 마지막에는 예측 score 를 얻어야됨 + 마지막 레이어에는 보통 선형변환을 씀 score를 위해
    def forward(self, X, A_norm):
        H = self.l1(X, A_norm)
        return self.l2(H)

    def train_gcn():
        set_seed(42)
        X, A_norm, y = load_karate()
        X # (N, D)
        A_norm # (N, N)
        y # (N)
        hidden_dim = 100
        input_dim = X.shape[-1]
        output_dim = 2
        model = GCN(input_dim, hidden_dim, output_dim)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
        loss_fn = nn.CrossEntropyLoss()
        for i in range(1000):
            optimizer.zero_grad()
            logits = model(X, A_norm)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()

if __name__ == "__main__":
    train_gcn()

In [ ]:
class GNNAtt(nn.Module):
    def __init__(self, node_dim, h_dim, 1):
        super().__init__()
        self.linear = nn.Linear(2*node_dim, h_dim)
        self.linear2 = nn.Linear(h_dim, 1)
        self.ReLU1 = nn.ReLU()
        self
    def forward(self,X, mask): #X:(N,dim)
        # mask: (N, N)
        # X : (N,dim)
        # node_pair(N,N,2*dim)
        # (N,N,1)
        # (N, N)
        N = X.shape[0]
        x1 = X.reshape(N, 1, -1) #(N, 1, dim)
        x2 = X.reshape(1, N, -1) #(1, N, dim)
        x1 = x1.expend(N, N ,-1) #(N, N, dim)
        x2 = x2.expendd(N, N, -1) #(N, N, dim)
        x_cat = torch.cat([x1,x2],dim=-1) #(N, N, 2*dim)
        s = self.linear2(self.ReLU1(self.linear(x_cat)))
        s = s.masked_fill(~mask, torch.tensor(float("-inf")))
        s = s.reshape(N, N)
        return torch.softmax(s, dim=-1)

class ATTGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.ReLU = nn.ReLU()
        self.a_l = GNNAtt(output, )

    def forward(self, X, mask):
        h = self.lin(X) # (N , out_dim)
        # X : (N, D)
        # A_norm : (N, N)
        # X - H :
        # W :
        a = self.a_l(h, mask)#(N,N)
        return self.ReLU(a @ h)

class ATTGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.l1 = ATTGCNLayer(in_dim, hidden_dim)
        self.l2 = nn.Linear(hidden_dim, out_dim)
        # a: (N, N)

        #레이어에 마지막에는 예측 score 를 얻어야됨 + 마지막 레이어에는 보통 선형변환을 씀 score를 위해
    def forward(self, X, A_norm):
        print(A_norm > 0)
        H = self.l1(X, A_norm  > 0)
        return self.l2(H)



NameError: name 'nn' is not defined

In [ ]:
class MultiGNNAttLayer(nn.Module):
    def __init__(self, in_dim, out_dim, n_heads):
        super().__init__()
        assert out_dim % n_heads == 0
        out_head_dim = out_dim // n_heads
        self.out_head_dim = out_head_dim
        self.lin = nn.ModuleList([nn.Linear(in_dim, out_head_dim) for _ in range(n_heads)])
        self.ReLU = nn.ReLU()
        self.a_l = nn.ModuleList([GNNAtt(out_head_dim, out_head_dim * 2) for _ in range(n_heads)])

    def forward(self, X, mask):
        out_list = []
        h = self.lin(X) # (N , out_dim)
        # X : (N, D)
        # A_norm : (N, N)
        # X - H :
        # W :
        a = self.a_l(h, mask)#(N,N)
        for _lin,_a_l in zip(self.lin, self.a_l):
            h = _lin(X) #(N,out_head_dim)
            a = _a_l(h, mask) #(N,N)
            out_list.append(self.ReLU(a @ h)) #(N , out_head_dim)
        # out_list : (n_heads ,N , out_head_dim)
        return torch.cat(out_list, dim=0) #(N ,out_dim = n_head)

class MultiGNNAtt(nn.Module):
    def __init__(self, node_dim, h_dim, 1):
        super().__init__()
        self.linear = nn.Linear(2*node_dim, h_dim)
        self.linear2 = nn.Linear(h_dim, 1)
        self.ReLU1 = nn.ReLU()
        self
    def forward(self,X, mask): #X:(N,dim)
        # mask: (N, N)
        # X : (N,dim)
        # node_pair(N,N,2*dim)
        # (N,N,1)
        # (N, N)
        N = X.shape[0]
        x1 = X.reshape(N, 1, -1) #(N, 1, dim)
        x2 = X.reshape(1, N, -1) #(1, N, dim)
        x1 = x1.expend(N, N ,-1) #(N, N, dim)
        x2 = x2.expendd(N, N, -1) #(N, N, dim)
        x_cat = torch.cat([x1,x2],dim=-1) #(N, N, 2*dim)
        s = self.linear2(self.ReLU1(self.linear(x_cat)))
        s = s.masked_fill(~mask, torch.tensor(float("-inf")))
        s = s.reshape(N, N)
        return torch.softmax(s, dim=-1)

class ATTGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.ReLU = nn.ReLU()
        self.a_l = GNNAtt(output, )

    def forward(self, X, mask):
        h = self.lin(X) # (N , out_dim)
        # X : (N, D)
        # A_norm : (N, N)
        # X - H :
        # W :
        a = self.a_l(h, mask)#(N,N)
        return self.ReLU(a @ h)

class ATTGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.l1 = ATTGCNLayer(in_dim, hidden_dim)
        self.l2 = nn.Linear(hidden_dim, out_dim)
        # a: (N, N)

        #레이어에 마지막에는 예측 score 를 얻어야됨 + 마지막 레이어에는 보통 선형변환을 씀 score를 위해
    def forward(self, X, A_norm):
        print(A_norm > 0)
        H = self.l1(X, A_norm  > 0)
        return self.l2(H)

